In [ ]:
# 1. Clone repository & system dependencies
%cd /content
!rm -rf /content/subtitlesssss
!git clone https://github.com/aananyaph/subtitlesssss
%cd /content/subtitlesssss

# Compatibility edits from original repository
!sed -i 's/^from typing import Optional$/from typing import Any, Callable, Dict, Optional, Tuple, Union/' /content/subtitlesssss/asr/indic_conformer.py 2>/dev/null || true


In [ ]:
# 2. Install required IndicTrans2, ONNX-GPU, and benchmarking packages
# Pre-install cython and wheel for IndicTransToolkit
!pip install --no-cache-dir wheel cython

# Install backend packages (note: faster-whisper pulls in CPU onnxruntime by default)
!pip install faster-whisper websockets jiwer scipy sentencepiece sacremoses indic-nlp-library sacrebleu pyngrok

# Force install ONNX Runtime GPU (CUDA 12 build)
!pip uninstall -y onnxruntime onnxruntime-gpu 2>/dev/null || true
!pip install --no-cache-dir onnxruntime-gpu --extra-index-url https://aiinfra.pkgs.visualstudio.com/PublicPackages/_packaging/onnxruntime-cuda-12/pypi/simple/

# Install IndicTransToolkit directly from AI4Bharat
!pip install --no-cache-dir --no-deps git+https://github.com/AI4Bharat/IndicTransToolkit.git


In [ ]:
# 3. Verify CUDA, PyTorch, and ONNX Runtime GPU Providers
!nvidia-smi
import os
import sys
import types
import torch
from google.colab import userdata

# Compatibility fixes for Hugging Face transformers >= 4.45
# 1. Hugging Face removed transformers.onnx in newer releases, but IndicTrans2
#    remote custom modeling code imports `from transformers.onnx import ...`.
#    We stub transformers.onnx to maintain backwards compatibility without errors.
try:
    import transformers
    import transformers.configuration_utils
    if not hasattr(transformers, "onnx"):
        dummy_onnx = types.ModuleType("transformers.onnx")
        dummy_onnx.OnnxConfig = object
        dummy_onnx.OnnxConfigWithPast = object
        dummy_onnx.OnnxSeq2SeqConfigWithPast = object
        sys.modules["transformers.onnx"] = dummy_onnx
        transformers.onnx = dummy_onnx
except Exception as e:
    print(f"Transformers ONNX compatibility shim warning: {e}")

import onnxruntime as ort
print("
--- System & Framework Verification ---")
print("Python Version:       ", sys.version.split()[0])
print("PyTorch Version:      ", torch.__version__)
print("CUDA Available:       ", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device:           ", torch.cuda.get_device_name(0))
    print("VRAM Allocated:       ", f"{torch.cuda.memory_allocated() / (1024**2):.1f} MB")
print("ONNX Runtime Version: ", ort.__version__)
print("ONNX Execution Provs: ", ort.get_available_providers())

assert torch.cuda.is_available(), "CRITICAL: CUDA is not available. Please switch Colab Runtime to GPU (T4)."
assert 'CUDAExecutionProvider' in ort.get_available_providers(), "CRITICAL: CUDAExecutionProvider is missing from ONNX Runtime!"
print("✅ All GPU runtimes verified successfully.")


In [ ]:
# 4. Pre-download IndicTrans2 1B repos and IndicConformer
import os, sys, shutil, pathlib
from huggingface_hub import snapshot_download, login

# Load Hugging Face access token from Colab Secrets or Environment
hf_token = None
try:
    from google.colab import userdata
    # Check common secret names (including user's exact name in screenshot: 'HF_TOKE')
    for candidate in ['HF_TOKEN', 'HF_TOKE', 'HUGGINGFACE_TOKEN', 'HF']:
        try:
            val = userdata.get(candidate)
            if val and val.strip():
                hf_token = val.strip()
                print(f"✅ Loaded Hugging Face token from Colab Secret: '{candidate}'")
                break
        except Exception:
            pass
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get('HF_TOKEN') or os.environ.get('HF_TOKE') or None

if not hf_token:
    print("⚠️ HF_TOKEN was not found in Colab Secrets (the 🔑 icon on the left panel).")
    try:
        entered = input("Paste your Hugging Face Token (or press Enter to skip): ").strip()
        if entered:
            hf_token = entered
    except Exception:
        pass

if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
    try:
        login(token=hf_token, add_to_git_credential=False)
        print("✅ Logged into Hugging Face successfully.")
    except Exception as e:
        print(f"⚠️ Hugging Face login warning: {e}")
else:
    print("⚠️ Running without HF_TOKEN. Gated models may fail with 401 error.")

# Pre-download IndicConformer for Indic ASR (Kannada/Hindi audio)
try:
    print("
Downloading IndicConformer ASR model (ai4bharat/indic-conformer-600m-multilingual)...")
    snapshot_download('ai4bharat/indic-conformer-600m-multilingual', token=hf_token)
    print("✅ IndicConformer snapshot ready.")

    # Fix ONNX Runtime 1.24+ symlink restriction:
    # Convert all symlinks in cache to real physical files to prevent 'escapes model directory' error
    cache_root = pathlib.Path(os.path.expanduser('~/.cache/huggingface/hub/models--ai4bharat--indic-conformer-600m-multilingual'))
    if cache_root.exists():
        fixed_count = 0
        for p in list(cache_root.rglob('*')):
            if p.is_symlink():
                target = p.resolve()
                if target.is_file():
                    p.unlink()
                    shutil.copyfile(target, p)
                    fixed_count += 1
        print(f"✅ Converted {fixed_count} cache symlinks to real physical files for ONNX Runtime compatibility.")
except Exception as e:
    print(f"⚠️ Could not download IndicConformer: {e}")
    print("👉 If this model is gated, accept terms at: https://huggingface.co/ai4bharat/indic-conformer-600m-multilingual")

# Pre-download IndicTrans2 translation models
# Prioritizing indic-en-1B for Kannada/Hindi -> English subtitles
repos_1b = [
    'ai4bharat/indictrans2-indic-en-1B',    # Handles Kannada -> English & Hindi -> English
    'ai4bharat/indictrans2-en-indic-1B',    # Handles English -> Indic
    'ai4bharat/indictrans2-indic-indic-1B', # Handles Indic -> Indic
]

for repo in repos_1b:
    try:
        print(f"
Pre-downloading/verifying: {repo}...")
        snapshot_download(repo, token=hf_token)
        print(f"✅ Ready: {repo}")
    except Exception as e:
        print(f"⚠️ Could not download {repo}: {e}")
        print(f"👉 If this model is gated, accept terms at: https://huggingface.co/{repo}")


In [ ]:
# 5. Apply dynamic ASR config patch and extract ONNX fallback translator
import os, sys

root = '/content/subtitlesssss'
path = f'{root}/backend/app_bridge.py'

# 1. Verify / Apply dynamic ASR config in app_bridge.py
if os.path.exists(path):
    with open(path, 'r', encoding='utf-8') as f:
        s = f.read()

    if 'def apply_asr_config' in s:
        print("✅ app_bridge.py already has dynamic ASR config built-in.")
    else:
        helper = '''# ---------- Dynamic ASR config (from client HANDSHAKE) ----------
_current_cfg_key = None

def apply_asr_config(cfg):
    """Rebuild pipeline to match client's audio/subtitle language choice."""
    global pipeline, _current_cfg_key
    key = (
        cfg.get("model_name") or "base",
        cfg.get("indic_lang") or None,
        cfg.get("asr_language") or None,
        cfg.get("asr_task") or None,
    )
    if pipeline is not None and key == _current_cfg_key:
        logging.info(f"[Bridge] ASR config unchanged {key}; keeping pipeline")
        return
    logging.info(f"[Bridge] Switching ASR config -> model={key[0]} indic={key[1]} lang={key[2]} task={key[3]}")
    import main_app
    old = pipeline
    if old is not None:
        try:
            old.stop()
        except Exception as e:
            logging.warning(f"[Bridge] old pipeline stop failed: {e}")
    if key[1]:
        from indic_backend import load_indic_model
        load_indic_model()
    new = main_app.DesktopLiveSubtitlePipeline(
        overlay=BridgeOverlay(),
        model_name=key[0],
        indic_lang=key[1],
        asr_language=key[2],
        asr_task=key[3],
    )
    try:
        new.start()
    except Exception as e:
        logging.warning(f"[Bridge] new pipeline start failed (no audio device?) {e}")
    pipeline = new
    _current_cfg_key = key
    logging.info("[Bridge] ASR config applied")

'''
        marker = '# ---------- WebSocket handler ----------'
        if marker in s:
            s = s.replace(marker, helper + marker)
            old_handshake = '                    logging.info(f"[Bridge] HANDSHAKE {data}")\n'
            new_handshake = '''                    logging.info(f"[Bridge] HANDSHAKE {data}")
                    _cfg = data.get("asrConfig")
                    if _cfg:
                        try:
                            await asyncio.get_running_loop().run_in_executor(None, apply_asr_config, _cfg)
                        except Exception as e:
                            logging.error(f"[Bridge] apply_asr_config failed: {e}", exc_info=True)
                            await ws.send(json.dumps({"type": "final", "text": f"Server model load failed: {e}", "is_final": True}))
'''
            if old_handshake in s:
                s = s.replace(old_handshake, new_handshake)
            with open(path, 'w', encoding='utf-8') as f:
                f.write(s)
            print("✅ app_bridge.py patched with dynamic ASR config.")
        else:
            print("ℹ️ app_bridge.py structure is up to date.")

# 2. Verify translator_live.py (ONNX fallback)
live_path = f'{root}/translator_live.py'
if os.path.exists(live_path):
    print("✅ translator_live.py already present.")
else:
    pipeline_path = f'{root}/pipeline.py'
    if os.path.exists(pipeline_path):
        import ast
        src = open(pipeline_path, encoding='utf-8').read()
        lines = src.splitlines(keepends=True)
        chunks = []
        for node in ast.parse(src).body:
            if isinstance(node, ast.Assign) and any(getattr(t, 'id', None) == 'LANG_MAP' for t in node.targets):
                chunks.append(''.join(lines[node.lineno - 1:node.end_lineno]))
            elif isinstance(node, ast.ClassDef) and node.name == 'IndicTransONNXModel':
                chunks.append(''.join(lines[node.lineno - 1:node.end_lineno]))
        header = '''import json, logging
from pathlib import Path
import numpy as np
import onnxruntime as ort
from huggingface_hub import snapshot_download
from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
from indicnlp.tokenize import indic_detokenize
from indicnlp.transliterate.unicode_transliterate import UnicodeIndicTransliterator
from sacremoses import MosesDetokenizer, MosesTokenizer
from tokenizers import Tokenizer

logger = logging.getLogger("translator_live")

'''
        with open(live_path, 'w', encoding='utf-8') as f:
            f.write(header + ''.join(chunks))
        print("✅ translator_live.py extracted successfully.")
    else:
        print("⚠️ pipeline.py not found to extract translator_live.py.")

print("✅ Cell 4 execution completed with 0 errors.")


In [ ]:
# 6. Direct Model Benchmark (6 Directions, 20 Sentences Each)
!cd /content/subtitlesssss && git pull origin main

import importlib
import gc
import json
import os
import sys
import time
import ctypes
import numpy as np

sys.path.insert(0, '/content/subtitlesssss')

if 'translator_gpu' in sys.modules:
    importlib.reload(sys.modules['translator_gpu'])
if 'translation_proxy' in sys.modules:
    importlib.reload(sys.modules['translation_proxy'])

import sacrebleu
import torch
import translation_proxy as tp

def free_system_ram():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    try:
        ctypes.CDLL('libc.so.6').malloc_trim(0)
    except Exception:
        pass

# 20 parallel test sentences across English, Hindi, and Kannada
test_sentences = {
    'en': [
        'Hello, how are you today?',
        'Good morning to everyone present here.',
        'The weather is very pleasant this evening.',
        'Where is the nearest railway station?',
        'Can you please help me with this problem?',
        'We are developing a real-time subtitle translation system.',
        'Artificial intelligence is transforming communication across the world.',
        'I enjoy listening to classical Indian music while working.',
        'Please make sure to submit your assignment before Friday.',
        'The train will arrive at platform number three in ten minutes.',
        'India is a country with rich cultural heritage and diversity.',
        'What time does the library close on weekends?',
        'Health and education are fundamental rights for every citizen.',
        'She works as a software engineer in Bengaluru.',
        'Can you recommend a good book to read during vacation?',
        'The scientific conference starts tomorrow at nine in the morning.',
        'Yoga and meditation help maintain both mental and physical health.',
        'Renewable energy sources are essential for sustainable growth.',
        'Thank you very much for your kind support and cooperation.',
        'Have a wonderful day ahead and stay safe.'
    ],
    'hi': [
        'नमस्ते, आप आज कैसे हैं?',
        'यहाँ उपस्थित सभी लोगों को शुभ प्रभात।',
        'आज शाम का मौसम बहुत सुहावना है।',
        'निकटतम रेलवे स्टेशन कहाँ है?',
        'क्या आप कृपया इस समस्या में मेरी मदद कर सकते हैं?',
        'हम एक रीयल-टाइम उपशीर्षक अनुवाद प्रणाली विकसित कर रहे हैं।',
        'कृत्रिम बुद्धिमत्ता दुनिया भर में संचार को बदल रही है।',
        'मुझे काम करते समय भारतीय शास्त्रीय संगीत सुनना पसंद है।',
        'कृपया शुक्रवार से पहले अपना असाइनमेंट जमा करना सुनिश्चित करें।',
        'ट्रेन दस मिनट में प्लेटफॉर्म नंबर तीन पर पहुंचेगी।',
        'भारत समृद्ध सांस्कृतिक विरासत और विविधता वाला देश है।',
        'सप्ताहांत पर पुस्तकालय कितने बजे बंद होता है?',
        'स्वास्थ्य और शिक्षा प्रत्येक नागरिक के मौलिक अधिकार हैं।',
        'वह बेंगलुरु में एक सॉफ्टवेयर इंजीनियर के रूप में काम करती है।',
        'क्या आप छुट्टियों के दौरान पढ़ने के लिए किसी अच्छी किताब की सिफारिश कर सकते हैं?',
        'वैज्ञानिक सम्मेलन कल सुबह नौ बजे शुरू होगा।',
        'योग और ध्यान मानसिक और शारीरिक स्वास्थ्य दोनों को बनाए रखने में मदद करते हैं।',
        'सतत विकास के लिए नवीकरणीय ऊर्जा स्रोत आवश्यक हैं।',
        'आपके सहयोग और समर्थन के लिए बहुत-बहुत धन्यवाद।',
        'आपका दिन मंगलमय हो और सुरक्षित रहें।'
    ],
    'kn': [
        'ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?',
        'ಇಲ್ಲಿ ಉಪಸ್ಥಿತರಿರುವ ಎಲ್ಲರಿಗೂ ಶುಭೋದಯ.',
        'ಇಂದು ಸಂಜೆಯ ಹವಾಮಾನವು ತುಂಬಾ ಆಹ್ಲಾದಕರವಾಗಿದೆ.',
        'ಹತ್ತಿರದ ರೈಲ್ವೆ ನಿಲ್ದಾಣ ಎಲ್ಲಿದೆ?',
        'ದಯವಿಟ್ಟು ಈ ಸಮಸ್ಯೆಯನ್ನು ಬಗೆಹರಿಸಲು ನನಗೆ ಸಹಾಯ ಮಾಡಬಹುದೇ?',
        'ನಾವು ನೈಜ-ಸಮಯದ ಉಪಶೀರ್ಷಿಕೆ ಅನುವಾದ ವ್ಯವಸ್ಥೆಯನ್ನು ಅಭಿವೃದ್ಧಿಪಡಿಸುತ್ತಿದ್ದೇವೆ.',
        'ಕೃತಕ ಬುದ್ಧಿಮತ್ತೆಯು ಪ್ರಪಂಚದಾದ್ಯಂತ ಸಂವಹನವನ್ನು ಪರಿವರ್ತಿಸುತ್ತಿದೆ.',
        'ಕೆಲಸ ಮಾಡುವಾಗ ನನಗೆ ಭಾರತೀಯ ಶಾಸ್ತ್ರೀಯ ಸಂಗೀತವನ್ನು ಕೇಳಲು ಇಷ್ಟ.',
        'ದಯವಿಟ್ಟು ಶುಕ್ರವಾರದ ಮೊದಲು ನಿಮ್ಮ ಕಾರ್ಯಯೋಜನೆಯನ್ನು ಸಲ್ಲಿಸುವುದನ್ನು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಿ.',
        'ರೈಲು ಹತ್ತು ನಿಮಿಷಗಳಲ್ಲಿ ಮೂರನೇ ಪ್ಲಾಟ್‌ಫಾರ್ಮ್‌ಗೆ ಆಗಮಿಸಲಿದೆ.',
        'ಭಾರತವು ಶ್ರೀಮಂತ ಸಾಂಸ್ಕೃತಿಕ ಪರಂಪರೆ ಮತ್ತು ವೈವಿಧ್ಯತೆಯನ್ನು ಹೊಂದಿರುವ ದೇಶವಾಗಿದೆ.',
        'ವಾರಾಂತ್ಯದಲ್ಲಿ ಗ್ರಂಥಾಲಯವು ಯಾವ ಸಮಯಕ್ಕೆ ಮುಚ್ಚುತ್ತದೆ?',
        'ಆರೋಗ್ಯ ಮತ್ತು ಶಿಕ್ಷಣ ಪ್ರತಿಯೊಬ್ಬ ನಾಗರಿಕನ ಮೂಲಭೂತ ಹಕ್ಕುಗಳು.',
        'ಅವಳು ಬೆಂಗಳೂರಿನಲ್ಲಿ ಸಾಫ್ಟ್‌ವೇರ್ ಎಂಜಿನಿಯರ್ ಆಗಿ ಕೆಲಸ ಮಾಡುತ್ತಿದ್ದಾಳೆ.',
        'ರಜಾದಿನಗಳಲ್ಲಿ ಓದಲು ಉತ್ತಮ ಪುಸ್ತಕವನ್ನು ನೀವು ಶಿಫಾರಸು ಮಾಡಬಹುದೇ?',
        'ವೈಜ್ಞಾನಿಕ ಸಮ್ಮೇಳನವು ನಾಳೆ ಬೆಳಿಗ್ಗೆ ಒಂಬತ್ತು ಗಂಟೆಗೆ ಪ್ರಾರಂಭವಾಗುತ್ತದೆ.',
        'ಯೋಗ ಮತ್ತು ಧ್ಯಾನ ಮಾನಸಿಕ ಮತ್ತು ದೈಹಿಕ ಆರೋಗ್ಯವನ್ನು ಕಾಪಾಡಿಕೊಳ್ಳಲು ಸಹಾಯ ಮಾಡುತ್ತವೆ.',
        'ಸುಸ್ಥಿರ ಬೆಳವಣಿಗೆಗೆ ನವೀಕರಿಸಬಹುದಾದ ಇಂಧನ ಮೂಲಗಳು ಅತ್ಯಗತ್ಯ.',
        'ನಿಮ್ಮ ಪ್ರೀತಿಯ ಬೆಂಬಲ ಮತ್ತು ಸಹಕಾರಕ್ಕೆ ತುಂಬಾ ಧನ್ಯವಾದಗಳು.',
        'ನಿಮ್ಮ ಮುಂದಿನ ದಿನವು ಅದ್ಭುತವಾಗಿರಲಿ ಮತ್ತು ಸುರಕ್ಷಿತವಾಗಿರಿ.'
    ]
}

directions = [('en', 'hi'), ('hi', 'en'), ('en', 'kn'), ('kn', 'en'), ('hi', 'kn'), ('kn', 'hi')]
configs = [
    ('torch1b (greedy - low RAM)', 'torch1b', '1'),
    ('onnx (int8 distilled)', 'onnx', '1'),
]

benchmark_results = {}
print('==========================================================================')
print('BENCHMARK: 20 sentences per direction, 6 directions')
print('Comparing: torch1b greedy and onnx int8 distilled (RAM optimized <= 4GB)')
print('==========================================================================
')

for cfg_name, backend_mode, beam_val in configs:
    os.environ['TRANSLATOR_BACKEND'] = backend_mode
    os.environ['TR_BEAMS'] = beam_val
    tp.TRANSLATOR_BACKEND = backend_mode

    with tp._tr_lock:
        tp._model_cache.clear()
        tp._text_cache.clear()
        free_system_ram()
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()

    print(f'
>>> RUNNING CONFIG: {cfg_name} (backend={backend_mode}, beams={beam_val})')
    benchmark_results[cfg_name] = {}

    for src_iso, tgt_iso in directions:
        dir_key = f'{src_iso}->{tgt_iso}'
        sentences = test_sentences[src_iso]
        references = test_sentences[tgt_iso]
        latencies = []
        outputs = []

        # Warmup
        try:
            tp.translate_with_meta('Hello test.', src_iso, tgt_iso)
        except Exception:
            pass

        for s in sentences:
            t0 = time.perf_counter()
            out, _, _ = tp.translate_with_meta(s, src_iso, tgt_iso)
            latencies.append((time.perf_counter() - t0) * 1000.0)
            outputs.append(out)

        p50 = float(np.percentile(latencies, 50))
        p95 = float(np.percentile(latencies, 95))
        peak_vram_mb = (torch.cuda.max_memory_allocated() / (1024.0 * 1024.0)) if torch.cuda.is_available() else 0.0
        score_chrf = round(sacrebleu.corpus_chrf(outputs, [[r] for r in references], word_order=2).score, 2)

        benchmark_results[cfg_name][dir_key] = {
            'p50_ms': round(p50, 1),
            'p95_ms': round(p95, 1),
            'peak_vram_mb': round(peak_vram_mb, 1),
            'chrf2': score_chrf,
        }
        print(f'  [{dir_key}] p50: {p50:.1f}ms | p95: {p95:.1f}ms | Peak VRAM: {peak_vram_mb:.1f}MB | chrF++: {score_chrf}')

        # Reclaim host memory between directions
        free_system_ram()

print('
✅ Benchmark completed successfully! RAM stayed safely within limits.')


In [ ]:
# 7. Start ASR Bridge (8766) & Translation Proxy (8765)
import os
import time
import socket
import subprocess

os.makedirs('/content/subtitlesssss', exist_ok=True)

# 1. Cleanly kill any previous bridge or proxy instances
os.system('fuser -k -9 8765/tcp 8766/tcp 2>/dev/null || true')
os.system("pkill -9 -f 'app_bridge.py' 2>/dev/null || true")
os.system("pkill -9 -f 'translation_proxy.py' 2>/dev/null || true")
time.sleep(1)

# Helper function to check if a TCP port is open and accepting connections
def wait_for_port(port, timeout=30):
    start = time.time()
    while time.time() - start < timeout:
        with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
            s.settimeout(1.0)
            if s.connect_ex(('127.0.0.1', port)) == 0:
                return True
        time.sleep(0.5)
    return False

# 2. Start ASR Bridge on port 8766
print("Starting ASR Bridge on port 8766...")
bridge_log = open('/content/subtitlesssss/bridge.log', 'w', encoding='utf-8')
bridge_proc = subprocess.Popen(
    ['python', 'backend/app_bridge.py', '--port', '8766'],
    cwd='/content/subtitlesssss',
    stdout=bridge_log,
    stderr=subprocess.STDOUT
)

if wait_for_port(8766, timeout=30):
    print("✅ ASR Bridge is ONLINE on port 8766.")
else:
    print("⚠️ ASR Bridge did not bind port 8766 within 30s. Checking bridge.log:")
    os.system('tail -n 20 /content/subtitlesssss/bridge.log')

# 3. Start GPU Translation Proxy on port 8765
print("
Starting Translation Proxy on port 8765...")
proxy_log = open('/content/subtitlesssss/proxy.log', 'w', encoding='utf-8')

env = os.environ.copy()
env['UPSTREAM_PORT'] = '8766'
env['PROXY_PORT'] = '8765'
env['TRANSLATOR_BACKEND'] = 'torch1b'
env['TR_BEAMS'] = '1'  # Greedy for low latency and minimal RAM
env['MAX_RESIDENT_MODELS'] = '1'  # Keeps RAM footprint well below 12.7 GB

proxy_proc = subprocess.Popen(
    ['python', 'translation_proxy.py'],
    cwd='/content/subtitlesssss',
    env=env,
    stdout=proxy_log,
    stderr=subprocess.STDOUT
)

if wait_for_port(8765, timeout=30):
    print("✅ GPU Translation Proxy is ONLINE on port 8765.")
else:
    print("⚠️ Translation Proxy did not bind port 8765 within 30s. Checking proxy.log:")
    os.system('tail -n 20 /content/subtitlesssss/proxy.log')

print("
--- Current Service Status ---")
os.system('ss -tulpn | grep -E "8765|8766"')


In [ ]:
# 8. Start Pyngrok Tunnel (8765)
import os
import time
from pyngrok import conf, ngrok

# 1. Cleanly terminate any previous ngrok instances
os.system('pkill -9 -f ngrok 2>/dev/null || true')
try:
    ngrok.kill()
except Exception:
    pass
time.sleep(1)

# 2. Load NGROK_AUTHTOKEN from Colab Secrets, environment, or prompt
token = None
try:
    from google.colab import userdata
    for candidate in ['NGROK_AUTHTOKEN', 'NGROK_', 'NGROK_TOKEN', 'NGROK']:
        try:
            val = userdata.get(candidate)
            if val and val.strip():
                token = val.strip()
                print(f"✅ Loaded ngrok token from Colab Secret: '{candidate}'")
                break
        except Exception:
            pass
except Exception:
    pass

if not token:
    token = os.environ.get('NGROK_AUTHTOKEN') or os.environ.get('NGROK_') or None

if not token:
    print('⚠️ NGROK token was not found in Colab Secrets (the 🔑 icon on the left panel).')
    token = input('Paste your ngrok authtoken (from https://dashboard.ngrok.com/get-started/your-authtoken): ').strip() or None

# 3. Configure authtoken
if token:
    os.environ['NGROK_AUTHTOKEN'] = token
    conf.get_default().auth_token = token
    ngrok.set_auth_token(token)
    print('✅ Ngrok authtoken configured successfully.')

# 4. Connect tunnel using pyngrok Python API
domain = 'aspire-reluctant-obsession.ngrok-free.dev'
print(f'Connecting ngrok tunnel to proxy port 8765 (attempting static domain "{domain}")...')

tunnel = None
try:
    tunnel = ngrok.connect(8765, 'http', domain=domain)
except Exception as e:
    print(f'ℹ️ Note: Static domain "{domain}" could not be bound ({e}).')
    print('Connecting with standard free dynamic ngrok tunnel...')

if tunnel is None:
    try:
        tunnel = ngrok.connect(8765, 'http')
    except Exception as e2:
        print(f'❌ Failed to start ngrok tunnel: {e2}')
        raise

public_url = tunnel.public_url
wss_url = public_url.replace('https://', 'wss://').replace('http://', 'ws://')

print('
========================================================')
print('✅ NGROK IS ONLINE!')
print('Public HTTPS URL:     ', public_url)
print('Client WebSocket URL: ', wss_url)
print('========================================================')
if domain not in public_url:
    print(f'👉 Copy & paste this Client WebSocket URL into your PC Control Panel:
   {wss_url}')
print('========================================================
')


In [ ]:
# 9. Verify GPU Translation Layer & Latency
import os
import sys
import time

sys.path.insert(0, '/content/subtitlesssss')
import translation_proxy as tp

print('
--- Testing Direct GPU Translation through Proxy Layer ---')
test_cases = [
    ('hi', 'en', 'नमस्ते, आप आज कैसे हैं?'),
    ('kn', 'en', 'ನಮಸ್ಕಾರ, ನೀವು ಇಂದು ಹೇಗಿದ್ದೀರಿ?'),
    ('en', 'hi', 'Good morning, welcome to our presentation.'),
]

for src, tgt, txt in test_cases:
    t0 = time.perf_counter()
    trans, latency, cached = tp.translate_with_meta(txt, src, tgt)
    dt_ms = (time.perf_counter() - t0) * 1000.0
    print(f"[{src} -> {tgt}] Input:  {txt}")
    print(f"           Output: {trans}")
    print(f"           Latency: {dt_ms:.1f} ms (cached: {cached})
")

print("✅ GPU Translation Layer verified.")


In [ ]:
# 10. Live Logs Monitor
!grep -E 'SUB |translation layer|new upstream|HANDSHAKE|error|FAIL' /content/subtitlesssss/proxy.log /content/subtitlesssss/bridge.log 2>/dev/null | tail -n 30
